In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
os.environ["OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["OPENAI_DEPLOYMENT_NAME"] = "gpt-4o-mini"
os.environ["OPENAI_MODEL"] = "gpt-4o-mini"

open_api_key = os.getenv("AZURE_OPENAI_API_KEY")
if os.getenv("AZURE_OPENAI_API_KEY"):
    print(f"Azure Open API Key loaded: {open_api_key[:10]}...{open_api_key[-4:]}")
    print(f"Model: {os.getenv('OPENAI_MODEL')}")
    os.environ["OPENAI_API_KEY"] = open_api_key
else:
    print("Azure API Key not found in environment variables")
    print("Available environment variables with 'Azure':", [k for k in os.environ.keys() if 'Azure' in k.upper()])

groq_api_key = os.getenv("GROQ_API_KEY")
if os.getenv("GROQ_API_KEY"):
    print(f"GROQ API Key loaded: {groq_api_key[:10]}...{groq_api_key[-4:]}")
    os.environ["GROQ_API_KEY"] = groq_api_key
else:
    print("GROQ API Key not found in environment variables")
    print("Available environment variables with 'GROQ':", [k for k in os.environ.keys() if 'GROQ' in k.upper()])

tavily_api_key = os.getenv("TAVILY_API_KEY")
if tavily_api_key:
    print(f"Tavily API Key loaded: {tavily_api_key[:10]}...{tavily_api_key[-4:]}")
else:
    print("Tavily API Key not found in environment variables")
    print("Available environment variables with 'TAVILY':", [k for k in os.environ.keys() if 'TAVILY' in k.upper()])

Azure Open API Key loaded: 1Sttw3VbMy...EzVq
Model: gpt-4o-mini
GROQ API Key loaded: gsk_kJ3nEw...OOCG
Tavily API Key loaded: tvly-dev-h...xnHL


In [24]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_core.tools import tool


In [25]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv)

api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=2, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=500)


In [37]:
arxiv.invoke("Attention is all you need ")


"Published: 2024-07-22\nTitle: Attention Is All You Need But You Don't Need All Of It For Inference of Large Language Models\nAuthors: Georgy Tyukin, Gbetondji J-S Dovonon, Jean Kaddour, Pasquale Minervini\nSummary: The inference demand for LLMs has skyrocketed in recent months, and serving\nmodels with low latencies remains challenging due to the quadratic input length\ncomplexity of the attention layers. In this work, we investigate the effect of\ndropping MLP and attention layers at inference time o"

In [27]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
print(wiki)

api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/zama/Desktop/code/Roadmap-To-AI-ML/langgraph/venv/lib/python3.12/site-packages/wikipedia/__init__.py'>, top_k_results=2, lang='en', load_all_available_meta=False, doc_content_chars_max=500)


In [28]:
wiki.invoke("what is machine learning?")

'Page: Machine learning\nSummary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions. Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.\nML fi'

In [33]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(
    api_key=os.getenv("TAVILY_API_KEY"),
    top_k=2,
    max_results=2,
    max_content_length=500
)
tavily

TavilySearch(max_results=2, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********')))

In [34]:
tavily.invoke("What is the latest in AI research?")

{'query': 'What is the latest in AI research?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'title': 'AI News | Latest AI News, Analysis & Events',
   'url': 'https://www.artificialintelligence-news.com/',
   'content': 'AI News reports on the latest artificial intelligence news and insights. Explore industry trends from the frontline of AI. ... Chatbots, Enterprise, Research, Virtual Assistants. June 18, 2025. Industries. Ericsson and AWS bet on AI to create self-healing networks. Amazon, Applications, Artificial Intelligence, Companies, Industries, Telecoms',
   'score': 0.6578248,
   'raw_content': None},
  {'title': 'Artificial Intelligence News -- ScienceDaily',
   'url': 'https://www.sciencedaily.com/news/computers_math/artificial_intelligence/',
   'content': 'Everything on AI including futuristic robots with artificial intelligence, computer models of human intelligence and more. ... Your source for the latest research news. Follow: Facebook X/Tw

In [35]:
tools = [arxiv, wiki, tavily]

In [36]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    api_key=os.getenv("GROQ_API_KEY")
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7eccb82cb5c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7eccb81545f0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [38]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain.schema import HumanMessage, SystemMessage, AIMessage

# tavily
response = llm_with_tools.invoke([
    HumanMessage(
        content="What is the recent AI news"
    )
]).tool_calls
response


[{'name': 'tavily_search',
  'args': {'query': 'recent AI news', 'topic': 'news'},
  'id': 'g2jb09kdf',
  'type': 'tool_call'}]

In [51]:
response = llm_with_tools.invoke([
    HumanMessage(
        content="What gold price in india"
    )
]).tool_calls
response

[{'name': 'wikipedia',
  'args': {'query': 'gold price in india'},
  'id': 'hfdmx8ny8',
  'type': 'tool_call'}]

In [56]:
response = llm_with_tools.invoke([
    HumanMessage(
        content="Share details on Attention is all you need paper"
    )
]).tool_calls
response

[{'name': 'arxiv',
  'args': {'query': 'Attention is all you need'},
  'id': 'cvt6eb6g8',
  'type': 'tool_call'}]